![logo](https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/docs/source/_figures/Helmholtz-AI.png?raw=true)

# Hands-On Session: Random Forest Model Training

In this notebook, we build and evaluate a **Random Forest classifier** for predicting **AML vs. non-AML** using RNA-seq gene expression data from [Warnat-Herresthal et al. (2020)](https://doi.org/10.1016/j.isci.2019.100780), *"Scalable prediction of acute myeloid leukemia using high-dimensional machine learning and blood transcriptomics."*. The objectives of this notebook are to define an appropriate data splitting strategy, train a Random Forest model, evaluate its predictive performance, and critically assess its robustness and generalizability.

--------

## Getting Started

### Setup Colab environment

If you installed the packages and requirements on your own machine, you can skip this section and start from the import section.
Otherwise, you can follow and execute the tutorial on your browser. In order to start working on the notebook, click on the following button. This will open this page in the Colab environment and you will be able to execute the code on your own.

<a href="https://colab.research.google.com/github/HelmholtzAI-Consultants-Munich/XAI-Tutorials/blob/main/xai-for-random-forest/Bio-6-HandsOn_AML_RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Now that you opened the notebook in Colab, follow the next step:

1. Run this cell to connect your Google Drive to Colab and install packages
2. Allow this notebook to access your Google Drive files. Click on 'Yes', and select your account.
3. "Google Drive for desktop wants to access your Google Account". Click on 'Allow'.
   
At this point, a folder has been created in your Drive and you can navigate it through the lefthand panel in Colab, you might also have received an email that informs you about the access on your Google Drive.

In [1]:
# Mount drive folder to dbe abale to download repo
# from google.colab import drive
# drive.mount('/content/drive')

# Switch to correct folder'
# %cd /content/drive/MyDrive

In [2]:
# Don't run this cell if you already cloned the repo 
# %rm -r XAI-Tutorials
# !git clone --branch main https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials.git

In [3]:
# Install al required dependencies and package versions
# %cd XAI-Tutorials
# !pip install -r requirements_xai-for-random-forest.txt
# %cd xai-for-random-forest

### Imports

Let's start with importing all required Python packages.

In [1]:
# Load the required packages

import os
import sys
import pickle

import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score

Now, we fix the random seeds to ensure reproducible results, as we work with (pseudo) random numbers.

In [2]:
# assert reproducible random number generation
seed = 1
np.random.seed(seed)

## Dataset Description

The data originates from [Warnat-Herresthal et al. (2020)](https://doi.org/10.1016/j.isci.2019.100780), *"Scalable prediction of acute myeloid leukemia using high-dimensional machine learning and blood transcriptomics."*

The original study constructed a large multi-study dataset through a systematic search of the GEO database, followed by multiple filtering steps (cell type, species, platform, minimum sample size, and expert curation), as illustrated in teh figure below. After removal of duplicates and exclusion of studies using sorted cell populations, three platform-specific datasets were established.

In this hands-on session, we focus exclusively on **Dataset 3 (RNA-seq)**, which contains:

- **1,181 samples**
- **AML patients**
- **Non-AML samples**, including:
  - Healthy individuals  
  - Other leukemia subtypes (e.g., ALL, CLL, CML, MDS)  
  - Other non-leukemic diseases  

The dataset includes **log2-transformed, normalized gene expression values** for over **12,000 genes**. In the original study, all datasets were harmonized to a shared gene set (12,708 genes) to enable cross-platform comparisons. Each dataset was normalized independently per platform prior to downstream analysis. Gene expression profiling was performed on **Peripheral Blood Mononuclear Cells (PBMC)** and **Bone Marrow (BM)** samples. In the original study, these tissues were considered equivalent for AML diagnosis and analyzed jointly.

For this hands-on session, we use the normalized RNA-seq data as provided and focus on modelling and interpretability rather than preprocessing.

## Load Processed Data

In this step, we load the pre-processed RNA-Seq dataset prepared in the previous notebook. This dataset contains:
- Normalized and cleaned gene expression features (log₂-transformed)
- Curated metadata including Condition, Disease, Tissue, GSE, and sample_id
- Annotated binary labels distinguishing AML from other samples

In [3]:
output_dir = "aml_case_study"

In [4]:
# load previously stored data
with open(os.path.join(output_dir, "dataset3.pickle"), "rb") as handle:
    data = pickle.load(handle)

data.head()

,PAX8,CCL5,MMP14,DTX2P1-UPK3BP1-PMS2P11,BAD,PRPF8,CAPNS1,RPL35,EIF4G2,EIF3D,...,ACTB,GAPDH,MIR3648-2,MIR3648-1,sample_id,Dataset,GSE,Condition,Disease,Tissue
55,429.141908,679.596678,46.224306,937.959260,1608.761060,27720.632747,12820.855077,13999.225407,30957.224082,14112.165495,...,207249.750988,265747.017753,17.718712,17.718712,GSM1202460,3,Simon (GSE49601),0_Control,ALL,PBMC
56,342.484974,770.785271,184.880818,2558.886562,1188.066680,24220.254267,12869.839728,13935.471198,18645.105479,10982.375835,...,91774.333718,38670.101249,17.562749,17.562749,GSM1202461,3,Simon (GSE49601),0_Control,ALL,BM
57,346.029244,178.616938,119.287279,1148.201355,697.178980,18303.955572,7988.833487,4892.082001,28528.632335,8830.426446,...,158699.288796,82700.438196,6.077328,6.077328,GSM1202462,3,Simon (GSE49601),0_Control,ALL,PBMC
58,273.100560,105.789293,51.844630,857.681016,534.301040,13737.032555,4964.479248,21213.436597,20873.148863,7660.995693,...,81507.355855,113369.609393,1.480776,1.480776,GSM1202463,3,Simon (GSE49601),0_Control,ALL,BM
59,219.733952,815.029434,43.389352,951.904754,349.511528,13659.251313,4370.811721,7804.724654,38471.198612,5333.294820,...,69913.344447,50200.238258,3.196909,3.196909,GSM1202464,3,Simon (GSE49601),0_Control,ALL,BM


## Data Splitting Strategy

As outlined in [Warnat-Herresthal et al. (2020)](https://doi.org/10.1016/j.isci.2019.100780), the RNA-seq dataset comprises gene expression profiles from 23 independent studies sourced from the Gene Expression Omnibus (GEO). This high inter-study diversity introduces both biological variation and technical heterogeneity (e.g., batch effects).

When building machine learning models on such multi-study data, the strategy used to construct training and testing sets critically determines how well performance reflects true generalizability.

<font color='green'>

### Task 1: Define and Implement a Data Splitting Strategy

Using the information from the EDA notebook, define and implement an appropriate data splitting strategy for the AML vs. non-AML classification task.


Two common approaches are:

### Random Sampling
- Samples for training and testing are randomly drawn from the entire pool, mixing samples from all 23 studies.
- This approach does not account for study-specific biases or batch effects.
- Study-specific signals may be present in both training and testing sets, which can lead to overly optimistic performance estimates.

### Cross-Study Sampling
- Training and test sets are constructed such that no study appears in both.
- If any sample from a study is used for training, all other samples from that study are excluded from the test set.
- This mimics a realistic deployment scenario in which a trained model is applied to an entirely new cohort.
- This strategy requires sufficient representation of each class across multiple studies to avoid imbalanced splits.

In this case study, we apply a **cross-study sampling** strategy to obtain a more realistic estimate of cross-study generalizability, using a **Stratified Group K-Fold** approach to:

- Ensure stratification: maintain a balanced class distribution (e.g., cancer vs. control) in both training and testing sets.
- Enforce group exclusivity: each group (study) is assigned entirely to either the training or the test set, preventing data leakage between them.

Given the high-dimensional setting of the dataset (12,000+ genes and 1,181 samples), rigorous study-level separation is essential to mitigate overfitting and assess robustness beyond study-specific artifacts.

*Note: For hyperparameter optimization during cross-validation, we also employ stratified group K-fold splitting to preserve this study-level exclusivity throughout the modeling pipeline*.

<div>
<center><img src="https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/raw/main/docs/source/_figures/aml_case_study_2.jpg" width="800"/><center>
</div>

<font size=1> Source: [Warnat-Herresthal et al. (2020)](https://doi.org/10.1016/j.isci.2019.100780)

### Cross-Platform Evaluation

Beyond study-level heterogeneity, clinical pipelines must also cope with changes in technological platforms (e.g., microarray vs. RNA-seq). It is therefore important to assess whether predictive models generalize not only across independent studies, but also across different measurement technologies.

In a cross-platform evaluation setting:

- A model is trained using data from one platform (e.g., RNA-seq), based on independently normalized gene expression values.
- The trained model is then applied **as is** to data generated on a different platform (e.g., Affymetrix microarrays), without any further fine-tuning.
- No samples from the target platform are used during training.

This setting evaluates:

- Robustness to technical variation between platforms.
- Stability of learned gene expression patterns.
- Transferability of predictive signatures across measurement technologies.

Cross-platform evaluation represents an even stricter test of generalization than cross-study sampling and is particularly relevant for long-term clinical deployment, where technologies may evolve over time.

<div>
<center><img src="https://github.com/HelmholtzAI-Consultants-Munich/XAI-Tutorials/raw/main/docs/source/_figures/aml_case_study_3.jpg" width="800"/><center>
</div>

<font size=1> Source: [Warnat-Herresthal et al. (2020)](https://doi.org/10.1016/j.isci.2019.100780)

### Define Training and Test Set

To enable robust and generalizable model evaluation, we construct a **train-test split that respects both stratification and study-level grouping**. This is especially important in high-dimensional transcriptomic data, where samples from the same study may introduce **study-specific biases (batch effects)**.

We use `StratifiedGroupKFold` to split the data, ensuring:

* **Class balance:** Training and test sets preserve similar proportions of AML and non-AML samples.
* **Study-level separation:** Samples from the same GSE (study) are never split across training and test sets, preventing data leakage and better reflecting generalization to unseen cohorts.

In [5]:
# define metadata and gene features
metadata = ["Condition", "Disease", "GSE", "sample_id", "Tissue", "Dataset"]
gene_features = data.drop(metadata, axis=1).columns

# define cross-validation strategy
cv = StratifiedGroupKFold(n_splits=5) # should approximate 80/20 split

# generate train, val and test sets
data_train_test = data.copy()
train_idx, test_idx = next(cv.split(data[gene_features], data["Condition"], data["GSE"]))
data_train_val, data_test = data.iloc[train_idx], data.iloc[test_idx]

print(f"Number of training samples {data_train_val.shape[0]} ({round(data_train_val.shape[0]/data_train_test.shape[0]*100)}%) with {sum(data_train_val['Condition']=='0_Control')} control and {sum(data_train_val['Condition']=='1_Cancer')} cancer samples and training studies: {list(set(data_train_val['GSE']))}")
print(f"Number of testing samples {data_test.shape[0]} ({round(data_test.shape[0]/data_train_test.shape[0]*100)}%) with {sum(data_test['Condition']=='0_Control')} control and {sum(data_test['Condition']=='1_Cancer')} cancer samples and testing studies: {list(set(data_test['GSE']))}")

Number of training samples 783 (66%) with 538 control and 245 cancer samples and training studies: ['GSE85712', 'Divinge (GSE58335)', 'Rojas.Pena (GSE67184)', 'Bouquet (GSE63085)', 'Wong (GSE79970)', 'Hoft (GSE87186)', 'Lavallee (GSE62190)', 'Lavallee (GSE52656)', 'Simon (GSE49601)', 'Lavallee (GSE66917)', 'Spinella (GSE78785)', 'Lavallee (GSE49642)', 'Dorr (GSE86884)', 'Garzon (GSE63646)', 'GSE72790', 'Meldi (GSE61162)', 'Chen (GSE32874)', 'Hoek (GSE64655)']
Number of testing samples 398 (34%) with 135 control and 263 cancer samples and testing studies: ['Lavallee (GSE67039)', 'Doss (GSE63703)', 'GSE81259', 'GSE63816', 'Henn (GSE45735)']


In [6]:
# split dataset into X and y and define groups with grouped cv

X_train_val = data_train_val[gene_features]
y_train_val = data_train_val["Condition"]
groups_train_val = data_train_val["GSE"]

X_test = data_test[gene_features]
y_test = data_test["Condition"]

In [7]:
with open(os.path.join(output_dir, "data_train_test.pickle"), 'wb') as handle:
    pickle.dump([data_train_val, data_test, gene_features], handle, protocol=pickle.HIGHEST_PROTOCOL)

**Details of the split:**

* The training set comprises **66% of the data (783 samples)**:

  * **538 control samples**
  * **245 cancer samples**
  * Drawn from **18 studies**

* The test set includes **34% of the data (398 samples)**:

  * **135 control samples**
  * **263 cancer samples**
  * Drawn from **5 unseen studies**, used exclusively for model evaluation

## Random Forest Classifier

Building on the defined data splitting strategy, the next step is to train a Random Forest model for AML vs. non-AML classification. Given the high-dimensional gene expression data and the multi-study structure of the dataset, the goal is to develop a model that achieves strong predictive performance while maintaining robust generalization.

<font color='green'>

### Task 2: Train a Random Forest Model

Using your previously defined data splitting strategy, train a Random Forest classifier for the AML vs. non-AML prediction task.

We train a Random Forest model to classify AML (cancer) vs. other (control) samples. To ensure optimal performance, we perform a grid search over key hyperparameters using Stratified Group K-Fold cross-validation, which preserves both class distribution and study separation. The parameter grid includes variations of tree depth, number of features, and leaf size. The best-performing model is selected based on validation accuracy and saved for reuse.

Finally, we evaluate the model on the training and independent test set using balanced accuracy, providing insight into both in-sample and out-of-sample performance. 

In [8]:
# define hyperparameter grid for Random Forest
grid = {"max_depth":[5, 10, 20], 
        "max_features": ['sqrt', 'log2'],
        "bootstrap": [True],
        "max_samples": [0.8],
        'n_estimators': [1000],
        'criterion': ['gini'],
        'min_samples_leaf': [1, 2, 5]
}

In [9]:
# define the cross-validation strategy
cv = StratifiedGroupKFold(n_splits=5)

# define the Random Forest classifier and a grid search with 5-fold stratified, grouped cross-validation and fit 
classifier = RandomForestClassifier(oob_score=True, random_state=42, n_jobs=4)
gridsearch_classifier = GridSearchCV(classifier, grid, cv=cv, scoring='accuracy', verbose=0)
gridsearch_classifier.fit(X_train_val, y_train_val, groups=groups_train_val)

# take the best estimator
rf_model = gridsearch_classifier.best_estimator_
print(gridsearch_classifier.best_params_)

# store the best model
with open(os.path.join(output_dir, 'rf_model.pickle'), 'wb') as handle:
    pickle.dump(rf_model, handle, protocol=pickle.HIGHEST_PROTOCOL)

{'bootstrap': True, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_samples': 0.8, 'min_samples_leaf': 2, 'n_estimators': 1000}


In [10]:
# check model performance
y_pred_train_val = rf_model.predict(X_train_val)
y_pred_test = rf_model.predict(X_test)

# is the model performing reasonably on the training data?
print(f'Model Performance on training data: {round(balanced_accuracy_score(y_train_val, y_pred_train_val)*100,2)} % macro accuracy.')

# is the model performing reasonably on the test data?
print(f'Model Performance on test data: {round(balanced_accuracy_score(y_test, y_pred_test)*100,2)} % macro accuracy and {round(f1_score(y_test, y_pred_test, average="macro")*100,2)} % macro F1 score.')

Model Performance on training data: 100.0 % macro accuracy.
Model Performance on test data: 97.19 % macro accuracy and 96.93 % macro F1 score.


## Conclusion

The Random Forest model demonstrates excellent predictive performance, achieving **100% balanced accuracy on the training data** and **97.19% balanced accuracy on the independent test set**.

The perfect training accuracy reflects the high flexibility of Random Forests in high-dimensional settings (12,000+ gene features and 1,181 samples). In such *p >> n* scenarios, models can easily fit the training data. Therefore, model quality should be assessed primarily based on performance on unseen data rather than training accuracy.

Balanced accuracy is reported to account for the moderate class imbalance between AML and non-AML samples, ensuring that both classes contribute equally to the evaluation metric.

Importantly, because training and test sets were separated at the **study level**, the reported test performance reflects genuine cross-study generalization rather than within-study memorization. This provides a more realistic estimate of how the model would perform when applied to data from entirely new studies.